In [1]:
import pandas as pd


In [2]:
emb_df = pd.read_parquet(
    "../data/complaint_embeddings.parquet"
)

emb_df.shape

(1375327, 4)

In [3]:
import numpy as np

embeddings = np.array(
    emb_df["embedding"].tolist(),
    dtype="float32"
)

embeddings.shape

(1375327, 384)

In [4]:
import faiss

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(embeddings)

using same embedding model

In [5]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

c:\Users\mijuu\Documents\mijuuhailu\intelligent-complaint-analysis\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1103.79it/s]


Build the Retriever

In [10]:
def retrieve_chunks(question, k=5):

    query_embedding = embedding_model.encode(
        [question]
    ).astype("float32")

    distances, indices = index.search(
        query_embedding,
        k
    )

    retrieved_docs = []

    for idx in indices[0]:

        retrieved_docs.append(
            emb_df.iloc[idx]
        )

    return retrieved_docs

In [15]:
results = retrieve_chunks(
    "Why are customers complaining about duplicate charges?"
)

for i, doc in enumerate(results, start=1):
    print(f"\n--- Result {i} ---")
    print(doc["document"][:300])


--- Result 1 ---
ents and supervisors, none of them sure why the charge has been duplicated. because it is through american express travel, american express classifies it as an internal charge. i can not dispute an internal charge with american express. hence the discussion with 20-30 agents, each directing me somew

--- Result 2 ---
they are charging overdraft fees to customers due to their peace of mind rebates!!!! i submitted a complaint on 2024 complaint id , that the submitted to citizen 's bank in which they responded it was a duplicate. it is not a duplicate. it may have appeared to thyem to be a duplicate because they re

--- Result 3 ---
o speak to anyone over the phone about any sort of account concern. the only method by which one can communicate with is via the help portion of their app through email. they have no telephone number at all nor any real time chat feature to hash out any problems. that's when i realized that the like

--- Result 4 ---
the merchant that the cha

prompt engineering 

In [13]:
PROMPT_TEMPLATE = """
You are a financial analyst assistant for CrediTrust.

Your task is to analyze customer complaints.

Use ONLY the information contained in the provided context.

If the answer cannot be determined from the context,
say that there is insufficient information.

Context:
{context}

Question:
{question}

Answer:
"""

In [16]:
retrieved_docs = retrieve_chunks(
    "Why are customers complaining about duplicate charges?"
)

In [18]:
context = "\n\n".join(
    [doc["document"] for doc in retrieved_docs]
)